
# WI-04 — Baseline Weather Forecasting

## 1. Introduction

This notebook establishes baseline forecasting models for the Weather Intelligence 2.0 system.

The purpose is to determine how accurately future weather conditions can be estimated using simple, interpretable forecasting approaches before introducing more complex machine-learning algorithms.

Baseline performance will provide a reference against which subsequent forecasting models can be evaluated.

## 2. Forecasting Objectives

This notebook focuses on two weather forecasting targets:

1. **Daily Precipitation (`PRECTOTCORR`)** — measured in mm/day.
2. **Daily Maximum Temperature (`T2M_MAX`)** — measured in °C.

Forecasts will be generated for seven future horizons:

- T+1
- T+2
- T+3
- T+4
- T+5
- T+6
- T+7

The initial spatial forecasting unit is the weather grid cell. Administrative district-level forecasting is outside the scope of this notebook.

## 3. Dataset

The canonical input is the leakage-safe feature dataset generated in WI-03.

**Dataset version:** `weather-features-v1`

**Dataset location:**

`phase3b_weather_intelligence/data/processed/weather_features_v1/weather-features-v1`

Dataset characteristics:

- 2,520 weather grid cells.
- 11,045,160 forecast-origin observations.
- 53 predictor features.
- 14 target columns.
- 252 Parquet batches.
- Seven forecast horizons for each of the two target variables.

The dataset includes current-day weather, historical lag features, rolling statistics, calendar features, and spatial coordinates.

## 4. Forecast Origin and Information Availability

The forecasting origin is defined as the end of day *t*, after observations for day *t* are available.

Every baseline must follow this information-availability rule.

No future observations may be used to generate a forecast.

## 5. Baseline Forecasting Methods

Three baseline approaches will be evaluated.

### 5.1 Persistence Baseline

The latest available weather observation is used as the forecast for every future horizon.

- Precipitation: repeat `PRECTOTCORR_current`.
- Maximum temperature: repeat `T2M_MAX_current`.

This baseline measures how well the latest observed conditions carry forward into the future.

### 5.2 Seasonal Naive Baseline

The corresponding calendar-date observation from the previous year is used as the forecast reference.

The source observation must be available at the forecast origin.

Leap-year and calendar-date matching must be handled explicitly. A fixed 365-day shift must not be assumed to represent the same calendar date in every year.

### 5.3 Training-period Climatology Baseline

A historical climatological reference will be calculated using training-period observations only.

The initial approach will use grid-cell and calendar-day historical averages.

Validation and final holdout observations must not contribute to climatology calculations.

## 6. Chronological Data Splitting

The chronological split established in WI-03 will be retained.

| Dataset | Period | Purpose |
|---|---|---|
| Training | 2013–2019 | Calculate baseline reference statistics |
| Validation | 2020–2022 | Evaluate and compare baseline forecasts |
| Final holdout | 2023–2024 | Reserved for final evaluation |

Forecast rows will be assigned independently for each horizon using the target date:

`target_date = forecast_origin_date + horizon_days`

Random splitting is prohibited.

The final holdout must remain untouched during baseline development and model selection.

## 7. Evaluation Metrics

Baseline performance will be evaluated separately for each target and forecast horizon.

### General regression metrics

- Mean Absolute Error (MAE).
- Root Mean Squared Error (RMSE).

### Additional precipitation evaluation

Because precipitation data contains many dry observations and a highly skewed distribution, overall regression metrics alone are insufficient.

Additional evaluation will include:

- Wet-day detection performance.
- Wet-day-only MAE.
- Rainfall quantile error.
- Performance across different rainfall intensity ranges.

The wet-day threshold and evaluation definitions must be documented before interpreting results.

## 8. Experimental Controls

The following controls apply throughout this notebook:

1. Use the canonical WI-03 feature dataset.
2. Preserve the forecast-origin availability convention.
3. Use training observations only when calculating climatology.
4. Do not use validation or holdout observations to construct training references.
5. Do not use the final holdout for baseline selection or tuning.
6. Do not introduce random train-test splitting.
7. Do not replace structural missing values with zero.
8. Do not modify the canonical feature dataset.
9. Record experiment configurations, evaluation metrics, and generated artifacts.
10. Report baseline performance independently for each target and horizon.

## 9. Expected Outputs

This notebook is expected to produce:

- Baseline experiment specification.
- Persistence forecast evaluation.
- Seasonal naive forecast evaluation.
- Training-period climatology evaluation.
- Target-wise and horizon-wise performance comparisons.
- Precipitation-specific evaluation results.
- Baseline experiment summary.
- Reproducible evaluation artifacts and reports.

## 10. Scope and Limitations

This notebook establishes historical forecasting benchmarks using the available weather archive.

The archive ends in 2024 and therefore does not provide operational live weather forecasts for 2026.

The baselines are intended as reference methods, not as production-ready weather forecasts.

The analysis is conducted at weather grid-cell level. Grid cells must not be interpreted as administrative districts without a validated spatial mapping.

No crop damage, disease outbreak, or yield impact conclusions will be inferred from these weather-only baseline forecasts.

## 11. Completion Criteria

WI-04 will be considered complete when:

- All three baseline methods have been implemented and verified.
- Forecast alignment has been checked for all seven horizons.
- No future information is used in forecast generation.
- Training-derived climatology is isolated from validation and holdout data.
- Validation performance has been evaluated for both targets.
- Precipitation-specific evaluation has been completed.
- Results and experiment artifacts have been saved and audited.

**Next stage:** WI-05 — Machine Learning-based Weather Forecasting.

---

**Notebook status:** Not started.


In [1]:

# ============================================================
# WI-04 — CELL 1
# Environment and Dataset Initialization
# ============================================================

from pathlib import Path
import json
import sys
import importlib.util
import pandas as pd
import numpy as np
import pyarrow
import pyarrow.parquet as pq

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

while (
    PROJECT_ROOT != PROJECT_ROOT.parent
    and not (PROJECT_ROOT / "phase3b_weather_intelligence").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "phase3b_weather_intelligence").exists(), (
    "Could not locate project root. Check notebook working directory."
)

PHASE3B_DIR = PROJECT_ROOT / "phase3b_weather_intelligence"

FEATURE_DATASET_DIR = (
    PHASE3B_DIR
    / "data"
    / "processed"
    / "weather_features_v1"
    / "weather-features-v1"
)

REPORT_DIR = PHASE3B_DIR / "reports" / "weather_intelligence_v2"
WI04_REPORT_DIR = REPORT_DIR / "WI_04_baseline_forecasting"

WI04_REPORT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Verify required files
# ------------------------------------------------------------

MANIFEST_PATH = FEATURE_DATASET_DIR / "batch_manifest.json"

assert FEATURE_DATASET_DIR.exists(), (
    f"Feature dataset directory not found: {FEATURE_DATASET_DIR}"
)

assert MANIFEST_PATH.exists(), (
    f"Batch manifest not found: {MANIFEST_PATH}"
)

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    batch_manifest = json.load(f)

print("Project root:", PROJECT_ROOT)
print("Feature dataset:", FEATURE_DATASET_DIR)
print("Report directory:", WI04_REPORT_DIR)

# ------------------------------------------------------------
# 3. Verify Parquet batches
# ------------------------------------------------------------

parquet_files = sorted(
    FEATURE_DATASET_DIR.rglob("batch_*.parquet")
)

assert len(parquet_files) == 252, (
    f"Expected 252 batches, found {len(parquet_files)}"
)

print("Parquet batches found:", len(parquet_files))

# ------------------------------------------------------------
# 4. Inspect dataset schema
# ------------------------------------------------------------

sample_schema = pq.read_schema(parquet_files[0])

print("\nFeature dataset schema:")
print(sample_schema)

assert len(sample_schema.names) == 69, (
    f"Expected 69 columns, found {len(sample_schema.names)}"
)

# ------------------------------------------------------------
# 5. Define experiment configuration
# ------------------------------------------------------------

TARGET_VARIABLES = ["PRECTOTCORR", "T2M_MAX"]
FORECAST_HORIZONS = list(range(1, 8))

TRAIN_START = "2013-01-01"
TRAIN_END = "2019-12-31"

VALIDATION_START = "2020-01-01"
VALIDATION_END = "2022-12-31"

HOLDOUT_START = "2023-01-01"
HOLDOUT_END = "2024-12-31"

# ------------------------------------------------------------
# 6. Verify required Python packages
# ------------------------------------------------------------

required_packages = [
    "numpy",
    "pandas",
    "pyarrow",
    "sklearn"
]

for package in required_packages:
    assert importlib.util.find_spec(package) is not None, (
        f"Required package missing: {package}"
    )

# ------------------------------------------------------------
# 7. Environment summary
# ------------------------------------------------------------

print("\nWI-04 INITIALIZATION: PASSED")
print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("PyArrow:", pyarrow.__version__)
print("Targets:", TARGET_VARIABLES)
print("Horizons:", FORECAST_HORIZONS)
print("Training:", TRAIN_START, "to", TRAIN_END)
print("Validation:", VALIDATION_START, "to", VALIDATION_END)
print("Final holdout:", HOLDOUT_START, "to", HOLDOUT_END)
print("Final holdout reserved: YES")
print("Full dataset loaded into memory: NO")


Project root: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence
Feature dataset: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\data\processed\weather_features_v1\weather-features-v1
Report directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_04_baseline_forecasting
Parquet batches found: 252

Feature dataset schema:
cell_id: string
date: timestamp[ns]
latitude: double
longitude: double
PRECTOTCORR_current: double
T2M_MAX_current: double
T2M_MIN_current: double
PRECTOTCORR_lag_1: double
PRECTOTCORR_lag_2: double
PRECTOTCORR_lag_3: double
PRECTOTCORR_lag_7: double
PRECTOTCORR_lag_14: double
PRECTOTCORR_lag_30: double
T2M_MAX_lag_1: double
T2M_MAX_lag_2: double
T2M_MAX_lag_3: double
T2M_MAX_lag_7: double
T2M_MAX_lag_14: double
T2M_MAX_lag_30: double
T2M_MIN_lag_1: double
T2M_MIN_lag_2: double
T2M_MIN_lag_3: double
T2M_MIN_lag_7: double
T2M_MIN_lag_14: double
T2M_MIN_l

In [2]:

# ============================================================
# WI-04 — CELL 2
# Baseline Experiment Configuration
# ============================================================

import json

BASELINE_SPEC = {
    "experiment_id": "WI-04",
    "experiment_name": "Baseline Weather Forecasting",
    "dataset_version": "weather-features-v1",
    "spatial_unit": "weather_grid_cell",
    "forecast_origin": (
        "End of day t, after day t weather observations are available"
    ),
    "targets": {
        "PRECTOTCORR": {
            "description": "Daily precipitation",
            "unit": "mm/day",
            "predictor": "PRECTOTCORR_current"
        },
        "T2M_MAX": {
            "description": "Daily maximum temperature",
            "unit": "degrees_Celsius",
            "predictor": "T2M_MAX_current"
        }
    },
    "horizons": FORECAST_HORIZONS,
    "splits": {
        "training": {
            "start": TRAIN_START,
            "end": TRAIN_END
        },
        "validation": {
            "start": VALIDATION_START,
            "end": VALIDATION_END
        },
        "final_holdout": {
            "start": HOLDOUT_START,
            "end": HOLDOUT_END
        }
    },
    "baselines": [
        "persistence",
        "seasonal_naive",
        "training_period_climatology"
    ],
    "evaluation_metrics": [
        "MAE",
        "RMSE"
    ],
    "precipitation_metrics": [
        "wet_day_detection",
        "wet_day_only_MAE",
        "rainfall_quantile_error"
    ],
    "climatology_policy": (
        "Calculate reference statistics using training-period "
        "observations only"
    ),
    "holdout_policy": (
        "Final holdout must not be used for baseline development, "
        "selection, or tuning"
    ),
    "random_split": False,
    "model_training_performed": False
}

BASELINE_SPEC_PATH = (
    WI04_REPORT_DIR / "WI_04_baseline_experiment_specification.json"
)

assert not BASELINE_SPEC_PATH.exists(), (
    "Specification already exists. Inspect it before overwriting."
)

with open(BASELINE_SPEC_PATH, "w", encoding="utf-8") as f:
    json.dump(BASELINE_SPEC, f, indent=2)

with open(BASELINE_SPEC_PATH, "r", encoding="utf-8") as f:
    verified_spec = json.load(f)

assert verified_spec["horizons"] == list(range(1, 8))
assert verified_spec["random_split"] is False
assert verified_spec["model_training_performed"] is False
assert len(verified_spec["baselines"]) == 3

print("BASELINE EXPERIMENT SPECIFICATION: SAVED AND VERIFIED")
print("Baselines:", verified_spec["baselines"])
print("Targets:", list(verified_spec["targets"].keys()))
print("Horizons:", verified_spec["horizons"])
print("Random split:", verified_spec["random_split"])
print("Model training:", verified_spec["model_training_performed"])
print("Saved:", BASELINE_SPEC_PATH)


BASELINE EXPERIMENT SPECIFICATION: SAVED AND VERIFIED
Baselines: ['persistence', 'seasonal_naive', 'training_period_climatology']
Targets: ['PRECTOTCORR', 'T2M_MAX']
Horizons: [1, 2, 3, 4, 5, 6, 7]
Random split: False
Model training: False
Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_04_baseline_forecasting\WI_04_baseline_experiment_specification.json


In [3]:

# ============================================================
# WI-04 — CELL 3
# Persistence Baseline Alignment Test
# ============================================================

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

sample_file = parquet_files[0]

sample_columns = [
    "cell_id",
    "date",
    "PRECTOTCORR_current",
    "T2M_MAX_current"
] + [
    f"{target}_target_t_plus_{h}"
    for target in TARGET_VARIABLES
    for h in FORECAST_HORIZONS
]

sample = pq.read_table(
    sample_file,
    columns=sample_columns
).to_pandas()

# ------------------------------------------------------------
# 1. Define persistence predictions
# ------------------------------------------------------------

for target, predictor in {
    "PRECTOTCORR": "PRECTOTCORR_current",
    "T2M_MAX": "T2M_MAX_current"
}.items():

    origin_values = sample[predictor].to_numpy()

    for horizon in FORECAST_HORIZONS:
        prediction = origin_values.copy()

        # Persistence forecast is constant across horizons.
        assert np.array_equal(
            prediction,
            origin_values,
            equal_nan=True
        )

        # Confirm target column exists and has expected length.
        target_col = f"{target}_target_t_plus_{horizon}"

        assert len(sample[target_col]) == len(prediction)

# ------------------------------------------------------------
# 2. Verify sample structure
# ------------------------------------------------------------

assert sample["cell_id"].notna().all()
assert sample["date"].notna().all()

assert sample["PRECTOTCORR_current"].notna().any()
assert sample["T2M_MAX_current"].notna().any()

# ------------------------------------------------------------
# 3. Show illustrative predictions
# ------------------------------------------------------------

example = sample[
    [
        "cell_id",
        "date",
        "PRECTOTCORR_current",
        "T2M_MAX_current"
    ]
].head(5).copy()

example["precipitation_persistence_T_plus_1_to_7"] = (
    example["PRECTOTCORR_current"]
)

example["temperature_persistence_T_plus_1_to_7"] = (
    example["T2M_MAX_current"]
)

print("PERSISTENCE ALIGNMENT TEST: PASSED")
print("Sample batch:", sample_file.name)
print("Sample rows:", len(sample))
print("Horizons verified:", FORECAST_HORIZONS)
print()
print("Illustrative persistence forecasts:")
print(example.to_string(index=False))


PERSISTENCE ALIGNMENT TEST: PASSED
Sample batch: batch_0000.parquet
Sample rows: 43830
Horizons verified: [1, 2, 3, 4, 5, 6, 7]

Illustrative persistence forecasts:
  cell_id       date  PRECTOTCORR_current  T2M_MAX_current  precipitation_persistence_T_plus_1_to_7  temperature_persistence_T_plus_1_to_7
cell_0000 2013-01-01                 1.63            28.39                                     1.63                                  28.39
cell_0000 2013-01-02                 0.63            27.45                                     0.63                                  27.45
cell_0000 2013-01-03                 0.27            27.81                                     0.27                                  27.81
cell_0000 2013-01-04                 0.79            27.89                                     0.79                                  27.89
cell_0000 2013-01-05                 0.21            28.21                                     0.21                                  28.21


In [4]:

# ============================================================
# WI-04 — CELL 4
# Persistence Baseline: Validation Evaluation
# ============================================================

import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.metrics import mean_absolute_error, mean_squared_error

persistence_results = []

validation_start = np.datetime64(VALIDATION_START, "D")
validation_end = np.datetime64(VALIDATION_END, "D")

predictor_map = {
    "PRECTOTCORR": "PRECTOTCORR_current",
    "T2M_MAX": "T2M_MAX_current"
}

# ------------------------------------------------------------
# Process each target and horizon
# ------------------------------------------------------------

for target, predictor in predictor_map.items():

    for horizon in FORECAST_HORIZONS:

        target_col = f"{target}_target_t_plus_{horizon}"

        total_rows = 0
        valid_rows = 0

        absolute_error_sum = 0.0
        squared_error_sum = 0.0

        for file_path in parquet_files:

            frame = pq.read_table(
                file_path,
                columns=[
                    "date",
                    predictor,
                    target_col
                ]
            ).to_pandas()

            origin_dates = pd.to_datetime(
                frame["date"]
            ).to_numpy(dtype="datetime64[D]")

            target_dates = (
                origin_dates + np.timedelta64(horizon, "D")
            )

            split_mask = (
                (target_dates >= validation_start)
                & (target_dates <= validation_end)
            )

            if not split_mask.any():
                continue

            subset = frame.loc[
                split_mask,
                [predictor, target_col]
            ]

            valid_mask = (
                subset[predictor].notna()
                & subset[target_col].notna()
            )

            valid = subset.loc[valid_mask]

            y_true = valid[target_col].to_numpy(dtype=np.float64)
            y_pred = valid[predictor].to_numpy(dtype=np.float64)

            assert np.isfinite(y_true).all()
            assert np.isfinite(y_pred).all()

            errors = y_true - y_pred

            total_rows += len(subset)
            valid_rows += len(valid)

            absolute_error_sum += np.abs(errors).sum()
            squared_error_sum += np.square(errors).sum()

            del frame, subset, valid, y_true, y_pred, errors

        assert valid_rows > 0
        assert valid_rows == total_rows, (
            f"Unexpected missing values: {target}, T+{horizon}"
        )

        mae = absolute_error_sum / valid_rows
        rmse = np.sqrt(squared_error_sum / valid_rows)

        persistence_results.append({
            "target": target,
            "horizon": f"T+{horizon}",
            "horizon_days": horizon,
            "validation_rows": valid_rows,
            "MAE": mae,
            "RMSE": rmse
        })

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

persistence_df = pd.DataFrame(persistence_results)

assert len(persistence_df) == 14
assert persistence_df["MAE"].notna().all()
assert persistence_df["RMSE"].notna().all()

print("PERSISTENCE VALIDATION EVALUATION: COMPLETED")
print()
print(
    persistence_df.to_string(
        index=False,
        formatters={
            "MAE": "{:.4f}".format,
            "RMSE": "{:.4f}".format,
            "validation_rows": "{:,}".format
        }
    )
)

# ------------------------------------------------------------
# Save artifacts
# ------------------------------------------------------------

csv_path = (
    WI04_REPORT_DIR / "WI_04_persistence_validation_metrics.csv"
)

json_path = (
    WI04_REPORT_DIR / "WI_04_persistence_validation_metrics.json"
)

assert not csv_path.exists(), "CSV already exists; inspect before replacing."
assert not json_path.exists(), "JSON already exists; inspect before replacing."

persistence_df.to_csv(csv_path, index=False)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        persistence_results,
        f,
        indent=2
    )

print()
print("Saved CSV:", csv_path)
print("Saved JSON:", json_path)
print("Final holdout accessed: NO")


PERSISTENCE VALIDATION EVALUATION: COMPLETED

     target horizon  horizon_days validation_rows    MAE    RMSE
PRECTOTCORR     T+1             1       2,761,920 3.2361  9.2107
PRECTOTCORR     T+2             2       2,761,920 4.0844 10.8430
PRECTOTCORR     T+3             3       2,761,920 4.3918 11.3489
PRECTOTCORR     T+4             4       2,761,920 4.5194 11.5325
PRECTOTCORR     T+5             5       2,761,920 4.6048 11.6422
PRECTOTCORR     T+6             6       2,761,920 4.6495 11.6799
PRECTOTCORR     T+7             7       2,761,920 4.6772 11.6909
    T2M_MAX     T+1             1       2,761,920 0.8810  1.3731
    T2M_MAX     T+2             2       2,761,920 1.1747  1.7891
    T2M_MAX     T+3             3       2,761,920 1.3380  2.0136
    T2M_MAX     T+4             4       2,761,920 1.4371  2.1477
    T2M_MAX     T+5             5       2,761,920 1.5100  2.2531
    T2M_MAX     T+6             6       2,761,920 1.5639  2.3380
    T2M_MAX     T+7             7       2,76

In [5]:

# ============================================================
# WI-04 — CELL 5
# Persistence Baseline: Precipitation Diagnostics
# ============================================================

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

WET_DAY_THRESHOLD_MM = 1.0

rainfall_diagnostics = []

intensity_bins = [
    (0.0, 1.0, "Below 1 mm"),
    (1.0, 5.0, "1–5 mm"),
    (5.0, 20.0, "5–20 mm"),
    (20.0, 50.0, "20–50 mm"),
    (50.0, float("inf"), "50+ mm")
]

for horizon in FORECAST_HORIZONS:

    target_col = f"PRECTOTCORR_target_t_plus_{horizon}"

    y_true_parts = []
    y_pred_parts = []

    for file_path in parquet_files:

        frame = pq.read_table(
            file_path,
            columns=[
                "date",
                "PRECTOTCORR_current",
                target_col
            ]
        ).to_pandas()

        origin_dates = pd.to_datetime(
            frame["date"]
        ).to_numpy(dtype="datetime64[D]")

        target_dates = (
            origin_dates + np.timedelta64(horizon, "D")
        )

        mask = (
            (target_dates >= np.datetime64(VALIDATION_START, "D"))
            & (target_dates <= np.datetime64(VALIDATION_END, "D"))
        )

        subset = frame.loc[
            mask,
            ["PRECTOTCORR_current", target_col]
        ]

        valid = subset.dropna()

        y_true_parts.append(
            valid[target_col].to_numpy(dtype=np.float64)
        )

        y_pred_parts.append(
            valid["PRECTOTCORR_current"].to_numpy(dtype=np.float64)
        )

        del frame, subset, valid

    y_true = np.concatenate(y_true_parts)
    y_pred = np.concatenate(y_pred_parts)

    assert np.isfinite(y_true).all()
    assert np.isfinite(y_pred).all()

    actual_wet = y_true >= WET_DAY_THRESHOLD_MM
    predicted_wet = y_pred >= WET_DAY_THRESHOLD_MM

    precision = precision_score(
        actual_wet, predicted_wet, zero_division=0
    )

    recall = recall_score(
        actual_wet, predicted_wet, zero_division=0
    )

    f1 = f1_score(
        actual_wet, predicted_wet, zero_division=0
    )

    wet_only_mae = (
        np.abs(y_true[actual_wet] - y_pred[actual_wet]).mean()
        if actual_wet.any()
        else np.nan
    )

    row = {
        "horizon": f"T+{horizon}",
        "wet_day_threshold_mm": WET_DAY_THRESHOLD_MM,
        "actual_wet_days": int(actual_wet.sum()),
        "predicted_wet_days": int(predicted_wet.sum()),
        "precision": precision,
        "recall": recall,
        "F1": f1,
        "actual_wet_day_MAE": wet_only_mae
    }

    for lower, upper, label in intensity_bins:
        band_mask = (
            (y_true >= lower)
            & (y_true < upper)
        )

        row[f"MAE_{label}"] = (
            np.abs(y_true[band_mask] - y_pred[band_mask]).mean()
            if band_mask.any()
            else np.nan
        )

    rainfall_diagnostics.append(row)

    del y_true, y_pred

diagnostics_df = pd.DataFrame(rainfall_diagnostics)

print("PERSISTENCE RAINFALL DIAGNOSTICS: COMPLETED")
print(
    diagnostics_df.to_string(
        index=False,
        formatters={
            col: "{:.4f}".format
            for col in diagnostics_df.select_dtypes(
                include="number"
            ).columns
            if col not in [
                "actual_wet_days",
                "predicted_wet_days"
            ]
        }
    )
)

diagnostics_path = (
    WI04_REPORT_DIR
    / "WI_04_persistence_rainfall_diagnostics.csv"
)

assert not diagnostics_path.exists()

diagnostics_df.to_csv(
    diagnostics_path,
    index=False
)

print("\nSaved:", diagnostics_path)
print("Final holdout accessed: NO")


PERSISTENCE RAINFALL DIAGNOSTICS: COMPLETED
horizon wet_day_threshold_mm  actual_wet_days  predicted_wet_days precision recall     F1 actual_wet_day_MAE MAE_Below 1 mm MAE_1–5 mm MAE_5–20 mm MAE_20–50 mm MAE_50+ mm
    T+1               1.0000          1158442             1158847    0.8219 0.8221 0.8220             6.9123         0.5803     3.3185      7.2234      17.5913    49.1145
    T+2               1.0000          1158442             1158997    0.7551 0.7555 0.7553             8.2326         1.0876     4.3578      8.1743      20.5874    58.5887
    T+3               1.0000          1158442             1158874    0.7262 0.7264 0.7263             8.5872         1.3609     4.6158      8.4204      21.4747    61.5446
    T+4               1.0000          1158442             1158804    0.7103 0.7106 0.7104             8.7012         1.4983     4.6978      8.4867      21.7201    63.0502
    T+5               1.0000          1158442             1158636    0.6996 0.6997 0.6997            

In [6]:

# ============================================================
# WI-04 — CELL 6
# Seasonal-Naive Calendar Alignment Verification
# ============================================================

import pandas as pd
import numpy as np

test_dates = pd.to_datetime([
    "2020-02-28",
    "2020-02-29",
    "2021-02-28",
    "2021-03-01",
    "2024-02-29",
    "2024-03-01"
])

def previous_year_calendar_date(date):
    previous_year = date.year - 1

    try:
        return date.replace(year=previous_year)
    except ValueError:
        # February 29 -> February 28 in a non-leap previous year
        return date.replace(year=previous_year, day=28)

seasonal_mapping = pd.DataFrame({
    "forecast_origin_date": test_dates
})

seasonal_mapping["previous_year_reference_date"] = (
    seasonal_mapping["forecast_origin_date"].apply(
        previous_year_calendar_date
    )
)

seasonal_mapping["reference_available_at_origin"] = (
    seasonal_mapping["previous_year_reference_date"]
    <= seasonal_mapping["forecast_origin_date"]
)

assert seasonal_mapping[
    "reference_available_at_origin"
].all()

# Specific leap-day expectations
assert (
    previous_year_calendar_date(
        pd.Timestamp("2020-02-29")
    ) == pd.Timestamp("2019-02-28")
)

assert (
    previous_year_calendar_date(
        pd.Timestamp("2024-02-29")
    ) == pd.Timestamp("2023-02-28")
)

print("SEASONAL-NAIVE CALENDAR ALIGNMENT: PASSED")
print(seasonal_mapping.to_string(index=False))


SEASONAL-NAIVE CALENDAR ALIGNMENT: PASSED
forecast_origin_date previous_year_reference_date  reference_available_at_origin
          2020-02-28                   2019-02-28                           True
          2020-02-29                   2019-02-28                           True
          2021-02-28                   2020-02-28                           True
          2021-03-01                   2020-03-01                           True
          2024-02-29                   2023-02-28                           True
          2024-03-01                   2023-03-01                           True


In [7]:

# ============================================================
# WI-04 — CELL 7
# Build Seasonal-Naive Reference Lookup
# ============================================================

import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
import pyarrow.parquet as pq

lookup_frames = []

lookup_columns = [
    "cell_id",
    "date",
    "PRECTOTCORR_current",
    "T2M_MAX_current"
]

for file_path in parquet_files:
    frame = pq.read_table(
        file_path,
        columns=lookup_columns
    ).to_pandas()

    frame["date"] = pd.to_datetime(frame["date"])

    lookup_frames.append(frame)

seasonal_lookup = pd.concat(
    lookup_frames,
    ignore_index=True
)

del lookup_frames

# ------------------------------------------------------------
# Validate unique cell/date keys
# ------------------------------------------------------------

assert not seasonal_lookup.duplicated(
    ["cell_id", "date"]
).any()

assert len(seasonal_lookup) == 11_045_160

# ------------------------------------------------------------
# Build a lookup key using calendar-date matching
# ------------------------------------------------------------

def previous_year_calendar_date(date):
    try:
        return date.replace(year=date.year - 1)
    except ValueError:
        return date.replace(year=date.year - 1, day=28)

seasonal_lookup["reference_date"] = (
    seasonal_lookup["date"].apply(
        previous_year_calendar_date
    )
)

# The reference lookup retains the actual date of the observation.
reference_table = seasonal_lookup[
    [
        "cell_id",
        "date",
        "PRECTOTCORR_current",
        "T2M_MAX_current"
    ]
].copy()

reference_table = reference_table.rename(
    columns={
        "date": "reference_date",
        "PRECTOTCORR_current": "PRECTOTCORR_seasonal_reference",
        "T2M_MAX_current": "T2M_MAX_seasonal_reference"
    }
)

# ------------------------------------------------------------
# Save reference table
# ------------------------------------------------------------

seasonal_lookup_path = (
    WI04_REPORT_DIR
    / "WI_04_seasonal_naive_reference.parquet"
)

assert not seasonal_lookup_path.exists(), (
    "Seasonal lookup already exists. Inspect before replacing."
)

reference_table.to_parquet(
    seasonal_lookup_path,
    index=False,
    compression="zstd"
)

# ------------------------------------------------------------
# Read-back verification
# ------------------------------------------------------------

check = pq.read_table(
    seasonal_lookup_path
).to_pandas()

assert len(check) == 11_045_160
assert not check.duplicated(
    ["cell_id", "reference_date"]
).any()

print("SEASONAL-NAIVE REFERENCE LOOKUP: SAVED")
print("Rows:", f"{len(check):,}")
print("Unique grid cells:", check["cell_id"].nunique())
print("Duplicate keys:", check.duplicated(
    ["cell_id", "reference_date"]
).sum())
print("Saved:", seasonal_lookup_path)


SEASONAL-NAIVE REFERENCE LOOKUP: SAVED
Rows: 11,045,160
Unique grid cells: 2520
Duplicate keys: 0
Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_04_baseline_forecasting\WI_04_seasonal_naive_reference.parquet


In [8]:

# ============================================================
# WI-04 — CELL 8
# Seasonal-Naive Baseline: Validation Evaluation
# ============================================================

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ------------------------------------------------------------
# 1. Load seasonal reference lookup
# ------------------------------------------------------------

reference_table = pq.read_table(
    seasonal_lookup_path
).to_pandas()

reference_table["reference_date"] = pd.to_datetime(
    reference_table["reference_date"]
)

reference_table = reference_table.set_index(
    ["cell_id", "reference_date"]
).sort_index()

assert reference_table.index.is_unique

# ------------------------------------------------------------
# 2. Calendar mapping
# ------------------------------------------------------------

def previous_year_calendar_date(date):
    try:
        return date.replace(year=date.year - 1)
    except ValueError:
        return date.replace(year=date.year - 1, day=28)

seasonal_results = []

validation_start = pd.Timestamp(VALIDATION_START)
validation_end = pd.Timestamp(VALIDATION_END)

for target in TARGET_VARIABLES:

    reference_col = (
        f"{target}_seasonal_reference"
    )

    for horizon in FORECAST_HORIZONS:

        target_col = f"{target}_target_t_plus_{horizon}"

        total_rows = 0
        valid_rows = 0
        absolute_error_sum = 0.0
        squared_error_sum = 0.0

        for file_path in parquet_files:

            frame = pq.read_table(
                file_path,
                columns=["cell_id", "date", target_col]
            ).to_pandas()

            frame["date"] = pd.to_datetime(frame["date"])

            frame["target_date"] = (
                frame["date"] + pd.to_timedelta(horizon, unit="D")
            )

            mask = (
                (frame["target_date"] >= validation_start)
                & (frame["target_date"] <= validation_end)
            )

            subset = frame.loc[mask].copy()

            if subset.empty:
                del frame, subset
                continue

            subset["reference_date"] = (
                subset["date"].apply(
                    previous_year_calendar_date
                )
            )

            # Reference must be available at forecast origin.
            assert (
                subset["reference_date"] <= subset["date"]
            ).all()

            keys = pd.MultiIndex.from_arrays(
                [
                    subset["cell_id"],
                    subset["reference_date"]
                ],
                names=["cell_id", "reference_date"]
            )

            references = reference_table.loc[
                keys, reference_col
            ].to_numpy()

            actual = subset[target_col].to_numpy(
                dtype=np.float64
            )

            valid = (
                np.isfinite(references)
                & np.isfinite(actual)
            )

            y_pred = references[valid]
            y_true = actual[valid]

            errors = y_true - y_pred

            total_rows += len(subset)
            valid_rows += len(y_true)

            absolute_error_sum += np.abs(errors).sum()
            squared_error_sum += np.square(errors).sum()

            del frame, subset, references, actual
            del valid, y_pred, y_true, errors

        assert total_rows == 2_761_920
        assert valid_rows == total_rows, (
            f"Missing seasonal reference: {target}, T+{horizon}"
        )

        mae = absolute_error_sum / valid_rows
        rmse = np.sqrt(squared_error_sum / valid_rows)

        seasonal_results.append({
            "target": target,
            "horizon": f"T+{horizon}",
            "horizon_days": horizon,
            "validation_rows": valid_rows,
            "MAE": mae,
            "RMSE": rmse
        })

# ------------------------------------------------------------
# 3. Save results
# ------------------------------------------------------------

seasonal_df = pd.DataFrame(seasonal_results)

assert len(seasonal_df) == 14
assert seasonal_df["MAE"].notna().all()
assert seasonal_df["RMSE"].notna().all()

print("SEASONAL-NAIVE VALIDATION EVALUATION: COMPLETED")
print(
    seasonal_df.to_string(
        index=False,
        formatters={
            "MAE": "{:.4f}".format,
            "RMSE": "{:.4f}".format,
            "validation_rows": "{:,}".format
        }
    )
)

csv_path = (
    WI04_REPORT_DIR
    / "WI_04_seasonal_naive_validation_metrics.csv"
)

json_path = (
    WI04_REPORT_DIR
    / "WI_04_seasonal_naive_validation_metrics.json"
)

assert not csv_path.exists()
assert not json_path.exists()

seasonal_df.to_csv(csv_path, index=False)

seasonal_df.to_json(
    json_path,
    orient="records",
    indent=2
)

print("\nSaved CSV:", csv_path)
print("Saved JSON:", json_path)
print("Final holdout used for evaluation: NO")


SEASONAL-NAIVE VALIDATION EVALUATION: COMPLETED
     target horizon  horizon_days validation_rows    MAE    RMSE
PRECTOTCORR     T+1             1       2,761,920 4.8002 12.1503
PRECTOTCORR     T+2             2       2,761,920 4.7911 12.1318
PRECTOTCORR     T+3             3       2,761,920 4.8055 12.1839
PRECTOTCORR     T+4             4       2,761,920 4.8382 12.2575
PRECTOTCORR     T+5             5       2,761,920 4.8639 12.3194
PRECTOTCORR     T+6             6       2,761,920 4.8695 12.3261
PRECTOTCORR     T+7             7       2,761,920 4.8473 12.2411
    T2M_MAX     T+1             1       2,761,920 1.9225  2.8616
    T2M_MAX     T+2             2       2,761,920 1.9292  2.8667
    T2M_MAX     T+3             3       2,761,920 1.9406  2.8826
    T2M_MAX     T+4             4       2,761,920 1.9518  2.8987
    T2M_MAX     T+5             5       2,761,920 1.9632  2.9156
    T2M_MAX     T+6             6       2,761,920 1.9727  2.9295
    T2M_MAX     T+7             7       2,

In [9]:

# ============================================================
# WI-04 — CELL 9
# Build Training-Only Grid-Cell Calendar-Day Climatology
# ============================================================

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

climatology_frames = []

climatology_columns = [
    "cell_id",
    "date",
    "PRECTOTCORR_current",
    "T2M_MAX_current"
]

for file_path in parquet_files:

    frame = pq.read_table(
        file_path,
        columns=climatology_columns
    ).to_pandas()

    frame["date"] = pd.to_datetime(frame["date"])

    # Training-period observations only.
    frame = frame[
        (frame["date"] >= TRAIN_START)
        & (frame["date"] <= TRAIN_END)
    ].copy()

    if not frame.empty:
        frame["calendar_day"] = (
            frame["date"].dt.strftime("%m-%d")
        )
        climatology_frames.append(frame)

    del frame

training_weather = pd.concat(
    climatology_frames,
    ignore_index=True
)

del climatology_frames

assert training_weather["date"].min() >= pd.Timestamp(TRAIN_START)
assert training_weather["date"].max() <= pd.Timestamp(TRAIN_END)

# ------------------------------------------------------------
# Calculate grid-cell and calendar-day means
# ------------------------------------------------------------

climatology_table = (
    training_weather
    .groupby(["cell_id", "calendar_day"], as_index=False)
    .agg(
        PRECTOTCORR_climatology=(
            "PRECTOTCORR_current", "mean"
        ),
        T2M_MAX_climatology=(
            "T2M_MAX_current", "mean"
        ),
        precipitation_observations=(
            "PRECTOTCORR_current", "count"
        ),
        temperature_observations=(
            "T2M_MAX_current", "count"
        )
    )
)

# ------------------------------------------------------------
# Leap-day policy
# ------------------------------------------------------------

# Use February 28 climatology as the February 29 fallback.
feb28 = climatology_table[
    climatology_table["calendar_day"] == "02-28"
].copy()

feb28["calendar_day"] = "02-29"

feb29_existing = climatology_table[
    climatology_table["calendar_day"] == "02-29"
]

feb29_cells = set(feb29_existing["cell_id"])
feb28_missing = feb28[
    ~feb28["cell_id"].isin(feb29_cells)
]

climatology_table = pd.concat(
    [climatology_table, feb28_missing],
    ignore_index=True
)

# ------------------------------------------------------------
# Validation checks
# ------------------------------------------------------------

assert not climatology_table.duplicated(
    ["cell_id", "calendar_day"]
).any()

assert climatology_table["cell_id"].nunique() == 2520

assert (
    climatology_table[
        climatology_table["calendar_day"] == "02-29"
    ].shape[0] == 2520
)

assert np.isfinite(
    climatology_table[
        ["PRECTOTCORR_climatology", "T2M_MAX_climatology"]
    ].to_numpy()
).all()

# ------------------------------------------------------------
# Save artifact
# ------------------------------------------------------------

climatology_path = (
    WI04_REPORT_DIR
    / "WI_04_training_period_climatology.parquet"
)

assert not climatology_path.exists(), (
    "Climatology artifact already exists. Inspect before replacing."
)

climatology_table.to_parquet(
    climatology_path,
    index=False,
    compression="zstd"
)

# Read-back verification
climatology_check = pq.read_table(
    climatology_path
).to_pandas()

assert len(climatology_check) == len(climatology_table)
assert climatology_check["cell_id"].nunique() == 2520
assert not climatology_check.duplicated(
    ["cell_id", "calendar_day"]
).any()

print("TRAINING-ONLY CLIMATOLOGY: SAVED AND VERIFIED")
print("Training observations:", f"{len(training_weather):,}")
print("Grid cells:", climatology_check["cell_id"].nunique())
print("Climatology records:", f"{len(climatology_check):,}")
print("Calendar days:", climatology_check["calendar_day"].nunique())
print("Leap-day fallback: February 28")
print("Validation data used: NO")
print("Holdout data used: NO")
print("Saved:", climatology_path)


TRAINING-ONLY CLIMATOLOGY: SAVED AND VERIFIED
Training observations: 6,441,120
Grid cells: 2520
Climatology records: 922,320
Calendar days: 366
Leap-day fallback: February 28
Validation data used: NO
Holdout data used: NO
Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_04_baseline_forecasting\WI_04_training_period_climatology.parquet


In [12]:

# Correct WI-04 report directory
REPORT_DIR = (
    Path(PROJECT_ROOT)
    / "phase3b_weather_intelligence"
    / "reports"
    / "weather_intelligence_v2"
    / "WI_04_baseline_forecasting"
)

CLIMATOLOGY_PATH = (
    REPORT_DIR / "WI_04_training_period_climatology.parquet"
)

assert REPORT_DIR.exists(), f"Report directory missing: {REPORT_DIR}"
assert CLIMATOLOGY_PATH.exists(), f"Climatology missing: {CLIMATOLOGY_PATH}"

print("WI-04 report directory corrected.")
print("Climatology file verified:", CLIMATOLOGY_PATH)


WI-04 report directory corrected.
Climatology file verified: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_04_baseline_forecasting\WI_04_training_period_climatology.parquet


In [14]:

from pathlib import Path
import json

FEATURE_DATASET_DIR = (
    Path(PROJECT_ROOT)
    / "phase3b_weather_intelligence"
    / "data"
    / "processed"
    / "weather_features_v1"
    / "weather-features-v1"
)

manifest_path = FEATURE_DATASET_DIR / "batch_manifest.json"

assert manifest_path.exists(), f"Manifest not found: {manifest_path}"

with open(manifest_path, "r", encoding="utf-8") as f:
    feature_manifest = json.load(f)

# Discover actual Parquet files; do not assume manifest key names.
FEATURE_BATCH_PATHS = sorted(FEATURE_DATASET_DIR.glob("batch_*.parquet"))

assert len(FEATURE_BATCH_PATHS) == 252, (
    f"Expected 252 feature batches, found {len(FEATURE_BATCH_PATHS)}"
)

assert all(path.exists() for path in FEATURE_BATCH_PATHS)

print("FEATURE BATCH PATHS: VERIFIED")
print("Feature dataset:", FEATURE_DATASET_DIR)
print("Manifest:", manifest_path)
print("Batch count:", len(FEATURE_BATCH_PATHS))
print("First batch:", FEATURE_BATCH_PATHS[0].name)
print("Last batch:", FEATURE_BATCH_PATHS[-1].name)


FEATURE BATCH PATHS: VERIFIED
Feature dataset: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\data\processed\weather_features_v1\weather-features-v1
Manifest: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\data\processed\weather_features_v1\weather-features-v1\batch_manifest.json
Batch count: 252
First batch: batch_0000.parquet
Last batch: batch_0251.parquet


In [15]:

# WI-04 — Cell 10: Training-only climatology validation evaluation

from pathlib import Path
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

CLIMATOLOGY_PATH = REPORT_DIR / "WI_04_training_period_climatology.parquet"

assert CLIMATOLOGY_PATH.exists(), "Training climatology file not found."

# Load the training-only climatology
climatology = pd.read_parquet(CLIMATOLOGY_PATH)

print("Climatology columns:", climatology.columns.tolist())

# Resolve expected climatology columns
def find_column(candidates, columns, description):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    raise KeyError(
        f"Could not identify {description}. "
        f"Available columns: {columns.tolist()}"
    )

clim_cell_col = find_column(
    ["cell_id"], climatology.columns, "cell ID column"
)
clim_day_col = find_column(
    ["calendar_day"], climatology.columns, "calendar-day column"
)

rain_clim_col = find_column(
    ["PRECTOTCORR_climatology", "PRECTOTCORR_mean"],
    climatology.columns,
    "rainfall climatology column"
)
temp_clim_col = find_column(
    ["T2M_MAX_climatology", "T2M_MAX_mean"],
    climatology.columns,
    "maximum-temperature climatology column"
)

climatology = climatology[
    [clim_cell_col, clim_day_col, rain_clim_col, temp_clim_col]
].copy()

climatology[clim_day_col] = climatology[clim_day_col].astype(str)

assert not climatology.duplicated(
    [clim_cell_col, clim_day_col]
).any(), "Duplicate climatology keys detected."

# Prepare lookup indexed by cell and calendar day
climatology = climatology.set_index(
    [clim_cell_col, clim_day_col]
)

# Identify target columns from the feature dataset
first_batch = pq.read_table(FEATURE_BATCH_PATHS[0]).schema.names

def find_target_column(target, horizon, columns):
    candidates = [
        f"{target}_target_t_plus_{horizon}",
        f"{target}_target_t+{horizon}",
        f"{target}_t_plus_{horizon}",
        f"{target}_t+{horizon}",
        f"{target}_target_{horizon}",
    ]
    for candidate in candidates:
        if candidate in columns:
            return candidate

    matches = [
        c for c in columns
        if target in c.lower()
        and str(horizon) in c
        and ("target" in c.lower())
    ]
    if len(matches) == 1:
        return matches[0]

    raise KeyError(
        f"Could not uniquely identify {target} T+{horizon} target. "
        f"Candidates found: {matches}"
    )

target_columns = {
    (target, horizon): find_target_column(
        target, horizon, first_batch
    )
    for target in TARGET_VARIABLES
    for horizon in FORECAST_HORIZONS
}

# Evaluate validation rows only.
# Validation membership is determined by target_date, not origin_date.
results = []

for target in TARGET_VARIABLES:
    clim_col = (
        rain_clim_col if target == "PRECTOTCORR"
        else temp_clim_col
    )

    for horizon in FORECAST_HORIZONS:
        target_col = target_columns[(target, horizon)]

        total_rows = 0
        missing_predictions = 0
        actual_values = []
        predicted_values = []

        for batch_path in FEATURE_BATCH_PATHS:
            df = pd.read_parquet(
                batch_path,
                columns=["cell_id", "date", target_col]
            )

            df["date"] = pd.to_datetime(df["date"])
            df["target_date"] = (
                df["date"] + pd.to_timedelta(horizon, unit="D")
            )

            validation_mask = (
                (df["target_date"] >= VALIDATION_START)
                & (df["target_date"] <= VALIDATION_END)
            )

            df = df.loc[validation_mask].copy()

            if df.empty:
                continue

            df = df.dropna(subset=[target_col])
            if df.empty:
                continue

            df["calendar_day"] = df["target_date"].dt.strftime("%m-%d")

            # Feb 29 fallback is already represented in the
            # training-only climatology table.
            lookup_index = pd.MultiIndex.from_arrays(
                [df["cell_id"], df["calendar_day"]],
                names=[clim_cell_col, clim_day_col]
            )

            predictions = climatology[clim_col].reindex(lookup_index)

            actual = df[target_col].to_numpy(dtype=float)
            predicted = predictions.to_numpy(dtype=float)

            total_rows += len(df)
            missing_predictions += int(np.isnan(predicted).sum())

            valid = np.isfinite(actual) & np.isfinite(predicted)

            actual_values.append(actual[valid])
            predicted_values.append(predicted[valid])

        if missing_predictions:
            raise AssertionError(
                f"{target} T+{horizon}: "
                f"{missing_predictions} validation rows lack climatology."
            )

        y_true = np.concatenate(actual_values)
        y_pred = np.concatenate(predicted_values)

        mae = float(np.mean(np.abs(y_true - y_pred)))
        rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

        results.append({
            "target": target,
            "horizon": f"T+{horizon}",
            "validation_rows": int(total_rows),
            "missing_climatology_predictions": int(missing_predictions),
            "MAE": mae,
            "RMSE": rmse,
        })

metrics_df = pd.DataFrame(results)

assert len(metrics_df) == 14
assert (metrics_df["missing_climatology_predictions"] == 0).all()

# Save validation metrics
csv_path = REPORT_DIR / "WI_04_climatology_validation_metrics.csv"
json_path = REPORT_DIR / "WI_04_climatology_validation_metrics.json"

metrics_df.to_csv(csv_path, index=False)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        metrics_df.to_dict(orient="records"),
        f,
        indent=2
    )

# Verify saved CSV
verified = pd.read_csv(csv_path)
assert len(verified) == 14

print("CLIMATOLOGY VALIDATION EVALUATION: COMPLETE")
print("Validation period:", VALIDATION_START, "to", VALIDATION_END)
print("Validation rows per target/horizon:", 
      metrics_df["validation_rows"].unique().tolist())
print("Missing climatology predictions:", 
      int(metrics_df["missing_climatology_predictions"].sum()))
print("\nValidation metrics:")
display(metrics_df.round(4))

print("\nSaved CSV:", csv_path)
print("Saved JSON:", json_path)
print("Final holdout 2023–2024 accessed: NO")


Climatology columns: ['cell_id', 'calendar_day', 'PRECTOTCORR_climatology', 'T2M_MAX_climatology', 'precipitation_observations', 'temperature_observations']
CLIMATOLOGY VALIDATION EVALUATION: COMPLETE
Validation period: 2020-01-01 to 2022-12-31
Validation rows per target/horizon: [2761920]
Missing climatology predictions: 0

Validation metrics:


,target,horizon,validation_rows,missing_climatology_predictions,MAE,RMSE
0,PRECTOTCORR,T+1,2761920,0,3.7649,9.0620
1,PRECTOTCORR,T+2,2761920,0,3.7649,9.0620
2,PRECTOTCORR,T+3,2761920,0,3.7649,9.0620
3,PRECTOTCORR,T+4,2761920,0,3.7649,9.0620
4,PRECTOTCORR,T+5,2761920,0,3.7649,9.0620
5,PRECTOTCORR,T+6,2761920,0,3.7649,9.0620
6,PRECTOTCORR,T+7,2761920,0,3.7649,9.0620
7,T2M_MAX,T+1,2761920,0,1.4997,2.2283
8,T2M_MAX,T+2,2761920,0,1.4997,2.2283
9,T2M_MAX,T+3,2761920,0,1.4997,2.2283



Saved CSV: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_04_baseline_forecasting\WI_04_climatology_validation_metrics.csv
Saved JSON: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_04_baseline_forecasting\WI_04_climatology_validation_metrics.json
Final holdout 2023–2024 accessed: NO


In [16]:

# WI-04 — Cell 11: Baseline validation comparison

import json
import pandas as pd
from pathlib import Path

REPORT_DIR = (
    Path(PROJECT_ROOT)
    / "phase3b_weather_intelligence"
    / "reports"
    / "weather_intelligence_v2"
    / "WI_04_baseline_forecasting"
)

metric_files = {
    "persistence": REPORT_DIR / "WI_04_persistence_validation_metrics.csv",
    "seasonal_naive": REPORT_DIR / "WI_04_seasonal_naive_validation_metrics.csv",
    "climatology": REPORT_DIR / "WI_04_climatology_validation_metrics.csv",
}

for name, path in metric_files.items():
    assert path.exists(), f"Missing {name} metrics: {path}"

def load_metrics(name, path):
    df = pd.read_csv(path)

    required = {"target", "horizon", "MAE", "RMSE"}
    missing = required - set(df.columns)
    assert not missing, f"{name} missing columns: {missing}"

    return df[["target", "horizon", "MAE", "RMSE"]].rename(
        columns={
            "MAE": f"{name}_MAE",
            "RMSE": f"{name}_RMSE",
        }
    )

comparison = load_metrics(
    "persistence", metric_files["persistence"]
)

for name in ["seasonal_naive", "climatology"]:
    comparison = comparison.merge(
        load_metrics(name, metric_files[name]),
        on=["target", "horizon"],
        how="inner",
        validate="one_to_one",
    )

assert len(comparison) == 14, (
    f"Expected 14 target/horizon comparisons, got {len(comparison)}"
)

mae_columns = [
    "persistence_MAE",
    "seasonal_naive_MAE",
    "climatology_MAE",
]

rmse_columns = [
    "persistence_RMSE",
    "seasonal_naive_RMSE",
    "climatology_RMSE",
]

comparison["best_by_MAE"] = (
    comparison[mae_columns]
    .idxmin(axis=1)
    .str.replace("_MAE", "", regex=False)
)

comparison["best_by_RMSE"] = (
    comparison[rmse_columns]
    .idxmin(axis=1)
    .str.replace("_RMSE", "", regex=False)
)

# Save detailed comparison
comparison_path = (
    REPORT_DIR / "WI_04_baseline_validation_comparison.csv"
)
summary_path = (
    REPORT_DIR / "WI_04_baseline_validation_comparison.json"
)

comparison.to_csv(comparison_path, index=False)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(
        comparison.to_dict(orient="records"),
        f,
        indent=2,
    )

# Aggregate winner counts
winner_counts = (
    comparison["best_by_MAE"]
    .value_counts()
    .rename_axis("baseline")
    .reset_index(name="MAE_wins")
)

print("BASELINE VALIDATION COMPARISON: COMPLETE")
print("\nDetailed comparison:")
display(comparison.round(4))

print("\nMAE winner counts:")
display(winner_counts)

print("\nSaved CSV:", comparison_path)
print("Saved JSON:", summary_path)
print("\nFinal holdout 2023–2024 accessed: NO")


BASELINE VALIDATION COMPARISON: COMPLETE

Detailed comparison:


,target,horizon,persistence_MAE,persistence_RMSE,seasonal_naive_MAE,seasonal_naive_RMSE,climatology_MAE,climatology_RMSE,best_by_MAE,best_by_RMSE
0,PRECTOTCORR,T+1,3.2361,9.2107,4.8002,12.1503,3.7649,9.0620,persistence,climatology
1,PRECTOTCORR,T+2,4.0844,10.8430,4.7911,12.1318,3.7649,9.0620,climatology,climatology
2,PRECTOTCORR,T+3,4.3918,11.3489,4.8055,12.1839,3.7649,9.0620,climatology,climatology
3,PRECTOTCORR,T+4,4.5194,11.5325,4.8382,12.2575,3.7649,9.0620,climatology,climatology
4,PRECTOTCORR,T+5,4.6048,11.6422,4.8639,12.3194,3.7649,9.0620,climatology,climatology
5,PRECTOTCORR,T+6,4.6495,11.6799,4.8695,12.3261,3.7649,9.0620,climatology,climatology
6,PRECTOTCORR,T+7,4.6772,11.6909,4.8473,12.2411,3.7649,9.0620,climatology,climatology
7,T2M_MAX,T+1,0.8810,1.3731,1.9225,2.8616,1.4997,2.2283,persistence,persistence
8,T2M_MAX,T+2,1.1747,1.7891,1.9292,2.8667,1.4997,2.2283,persistence,persistence
9,T2M_MAX,T+3,1.3380,2.0136,1.9406,2.8826,1.4997,2.2283,persistence,persistence



MAE winner counts:


,baseline,MAE_wins
0,climatology,9
1,persistence,5



Saved CSV: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_04_baseline_forecasting\WI_04_baseline_validation_comparison.csv
Saved JSON: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_04_baseline_forecasting\WI_04_baseline_validation_comparison.json

Final holdout 2023–2024 accessed: NO


In [17]:

# WI-04 — Cell 12: Baseline findings summary

import json
import pandas as pd
from pathlib import Path

REPORT_DIR = (
    Path(PROJECT_ROOT)
    / "phase3b_weather_intelligence"
    / "reports"
    / "weather_intelligence_v2"
    / "WI_04_baseline_forecasting"
)

comparison_path = (
    REPORT_DIR / "WI_04_baseline_validation_comparison.csv"
)
assert comparison_path.exists()

df = pd.read_csv(comparison_path)

baseline_columns = {
    "persistence": "persistence_MAE",
    "seasonal_naive": "seasonal_naive_MAE",
    "climatology": "climatology_MAE",
}

# Count validation MAE wins
win_counts = (
    df["best_by_MAE"]
    .value_counts()
    .to_dict()
)

# Identify winner for each target and horizon
winners = []
for _, row in df.iterrows():
    target_winner = min(
        baseline_columns,
        key=lambda name: row[baseline_columns[name]]
    )

    winners.append({
        "target": row["target"],
        "horizon": row["horizon"],
        "best_baseline_by_MAE": target_winner,
        "best_MAE": float(
            row[baseline_columns[target_winner]]
        ),
    })

summary = {
    "experiment": "WI-04 Baseline Weather Forecasting",
    "validation_period": "2020-01-01 to 2022-12-31",
    "training_period": "2013-01-01 to 2019-12-31",
    "final_holdout_period": "2023-01-01 to 2024-12-31",
    "targets": ["PRECTOTCORR", "T2M_MAX"],
    "forecast_horizons": [f"T+{h}" for h in range(1, 8)],
    "baselines": list(baseline_columns.keys()),
    "mae_win_counts": win_counts,
    "horizon_winners": winners,
    "findings": [
        "Climatology wins rainfall MAE from T+2 through T+7.",
        "Persistence wins rainfall MAE at T+1.",
        "Persistence wins T2M_MAX MAE from T+1 through T+4.",
        "Climatology wins T2M_MAX MAE from T+5 through T+7.",
        "Seasonal naive does not win any MAE comparison.",
        "Climatology is a seasonal expectation benchmark, not an event forecast."
    ],
    "holdout_accessed": False,
    "model_training_performed": False,
    "production_approval": False
}

output_path = REPORT_DIR / "WI_04_baseline_findings.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

with open(output_path, "r", encoding="utf-8") as f:
    verified = json.load(f)

assert verified["holdout_accessed"] is False
assert verified["model_training_performed"] is False
assert len(verified["horizon_winners"]) == 14

print("WI-04 BASELINE FINDINGS: SAVED AND VERIFIED")
print("MAE wins:", win_counts)
print("Target/horizon winners:", len(winners))
print("Saved:", output_path)
print("Final holdout accessed: NO")


WI-04 BASELINE FINDINGS: SAVED AND VERIFIED
MAE wins: {'climatology': 9, 'persistence': 5}
Target/horizon winners: 14
Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_04_baseline_forecasting\WI_04_baseline_findings.json
Final holdout accessed: NO
